## 1 - Project Setup
This section initializes the workspace, mounts Google Drive, extracts the dataset, and authenticates with HuggingFace Hub.

**Note:** DINOv3 checkpoints on HuggingFace are gated. You need to accept the license at
https://huggingface.co/facebook/dinov3-convnext-small-pretrain-lvd1689m and have an access
token with read permission before running the login cell below.

In [ ]:
!pip install -q transformers huggingface_hub accelerate

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import time
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from transformers import AutoModel
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, roc_curve, auc
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# Environment setup: works both on Colab (Drive-mounted) and on a local VM/JupyterLab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    DRIVE_ROOT = '/content/drive/MyDrive/internship-deepfake-forensic'
    DATASET_DIR = '/content/dataset'
    MODELS_DIR = os.path.join(DRIVE_ROOT, 'deepfake_models_final')

    if not os.path.exists(DATASET_DIR):
        print("Extracting the dataset... (this might take a few minutes)")
        !unzip -q {os.path.join(DRIVE_ROOT, 'deepfake_dataset.zip')} -d {DATASET_DIR}
        print("Extraction completed!")
    else:
        print("Dataset already present and ready to use!")
else:
    DATA_ROOT = '/home/lucia_pola/internship-deepfake-forensic/deepfake-forensics-pipeline/02_Extended_Framework'
    DATASET_DIR = os.path.join(DATA_ROOT, 'dataset')
    MODELS_DIR = os.path.join(DATA_ROOT, 'deepfake_models_final')
    print(f"Running outside Colab. Using local paths under: {DATA_ROOT}")

os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Dataset directory: {DATASET_DIR}")
print(f"Models directory:  {MODELS_DIR}")

In [ ]:
# Log in to HuggingFace Hub to access the gated DINOv3 checkpoints.
# This prompts for your access token interactively (paste it when asked).
!huggingface-cli login

## 2 - Dataset and Models Architecture

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, real_dirs, fake_dirs, transform=None):
        self.filepaths, self.labels = [], []
        self.transform = transform
        extensions = ('*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG')
        for d in real_dirs:
            for ext in extensions:
                paths = sorted(glob.glob(os.path.join(d, ext)))
                self.filepaths.extend(paths)
                self.labels.extend([0] * len(paths))
        for d in fake_dirs:
            for ext in extensions:
                paths = sorted(glob.glob(os.path.join(d, ext)))
                self.filepaths.extend(paths)
                self.labels.extend([1] * len(paths))

    def __len__(self): return len(self.filepaths)

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor([self.labels[idx]], dtype=torch.float32)


test_transforms = transforms.Compose([
    transforms.Resize((224, 224)), # Fail-safe resize
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
class StandardEfficientNet(nn.Module):
    def __init__(self, num_classes=1, pretrained=True):
        super().__init__()
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        self.model = models.efficientnet_b0(weights=weights)
        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Sequential(
            nn.Dropout(p=0.4, inplace=True),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.model(x)

class HybridDeepfakeDiscriminator(nn.Module):
    def __init__(self, num_classes=1, pretrained=True):
        super().__init__()
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        self.backbone = models.efficientnet_b0(weights=weights).features
        in_channels = 1280
        self.attention = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 8, 1, bias=False),
            nn.BatchNorm2d(in_channels // 8), nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 8, in_channels, 1, bias=False), nn.Sigmoid()
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4), nn.Linear(in_channels, 256), nn.ReLU(inplace=True),
            nn.Dropout(p=0.4), nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        features = features * self.attention(features)
        return self.classifier(torch.flatten(self.pool(features), 1))

class DINOv3ConvNeXtClassifier(nn.Module):
    """
    DINOv3 ConvNeXt-Small as a frozen Foundation-CNN feature extractor, with only a
    lightweight classification head trained on top. The backbone is kept fully frozen
    (see stepA1_baseline_dino.ipynb for the rationale).
    """
    def __init__(self, num_classes=1, model_id="facebook/dinov3-convnext-small-pretrain-lvd1689m"):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_id)
        for p in self.backbone.parameters():
            p.requires_grad = False

        hidden_sizes = getattr(self.backbone.config, "hidden_sizes", None)
        hidden_size = hidden_sizes[-1] if hidden_sizes is not None else self.backbone.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(hidden_size, num_classes)
        )

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()
        return self

    def state_dict(self, *args, **kwargs):
        # Checkpoints only ever contain the trainable head (see stepA1/stepA2)
        return self.classifier.state_dict(*args, **kwargs)

    def load_state_dict(self, state_dict, *args, **kwargs):
        return self.classifier.load_state_dict(state_dict, *args, **kwargs)

    def forward(self, x):
        with torch.no_grad():
            outputs = self.backbone(pixel_values=x)
            features = getattr(outputs, "pooler_output", None)
            if features is None:
                last_hidden = outputs.last_hidden_state
                features = last_hidden.mean(dim=[-2, -1]) if last_hidden.dim() == 4 else last_hidden.mean(dim=1)
        return self.classifier(features)

## 3 - Data Configuration

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in use: {device}")

path_baseline      = os.path.join(MODELS_DIR, 'baseline', 'path')
path_retrain       = os.path.join(MODELS_DIR, 'retrain', 'path')
path_baseline_dino = os.path.join(MODELS_DIR, 'baseline_dino', 'path')
path_retrain_dino  = os.path.join(MODELS_DIR, 'retrain_dino', 'path')
BATCH = 32
NUM_WORKERS = 4

ts_reals = [f'{DATASET_DIR}/FEI_MORPHV2_DATASET/test/original/', f'{DATASET_DIR}/CELEBDFV3_DATASET/test/original/']
ts_fakes = [f'{DATASET_DIR}/FEI_MORPHV2_DATASET/test/fake/', f'{DATASET_DIR}/CELEBDFV3_DATASET/test/fake/']

# Test Dataloader
test_loader = DataLoader(DeepfakeDataset(ts_reals, ts_fakes, test_transforms), BATCH, shuffle=False, num_workers=NUM_WORKERS)

## 4 - Load Models

In [ ]:
print("Loading Models for Direct Comparison...")

#Standard EfficientNet-B0
model_std_base = StandardEfficientNet(pretrained=False).to(device)
model_std_base.load_state_dict(torch.load(os.path.join(path_baseline, "std_b0_step1.pth"), map_location=device))

model_std_adv = StandardEfficientNet(pretrained=False).to(device)
model_std_adv.load_state_dict(torch.load(os.path.join(path_retrain, "std_b0_step2.pth"), map_location=device))

#Hybrid EfficientNet-B0
model_hyb_base = HybridDeepfakeDiscriminator(pretrained=False).to(device)
model_hyb_base.load_state_dict(torch.load(os.path.join(path_baseline, "hyb_b0_step1.pth"), map_location=device))

model_hyb_adv = HybridDeepfakeDiscriminator(pretrained=False).to(device)
model_hyb_adv.load_state_dict(torch.load(os.path.join(path_retrain, "hyb_b0_step2.pth"), map_location=device))

#DINOv3 ConvNeXt-Small (Foundation CNN, frozen backbone)
model_dino_base = DINOv3ConvNeXtClassifier().to(device)
model_dino_base.load_state_dict(torch.load(os.path.join(path_baseline_dino, "dino_convnext_small_step1.pth"), map_location=device))

model_dino_adv = DINOv3ConvNeXtClassifier().to(device)
model_dino_adv.load_state_dict(torch.load(os.path.join(path_retrain_dino, "dino_convnext_small_step2.pth"), map_location=device))

trained_models = {
    "Std EfficientNet-B0 (Step 1)": model_std_base,
    "Std EfficientNet-B0 (Step 2)": model_std_adv,
    "Hyb EfficientNet-B0 (Step 1)": model_hyb_base,
    "Hyb EfficientNet-B0 (Step 2)": model_hyb_adv,
    "DINOv3 ConvNeXt-S (Step 1)": model_dino_base,
    "DINOv3 ConvNeXt-S (Step 2)": model_dino_adv,
}

print("Models loaded and ready for the showdown!")

## 5 - Test Engine

In [ ]:
def get_all_predictions(model, loader, device):
    model.eval()
    all_labels, all_probs, all_preds = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images).view(-1)

            probs = torch.sigmoid(outputs)
            preds = (outputs > 0.0).float()

            all_labels.extend(labels.view(-1).cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    return np.array(all_labels), np.array(all_probs), np.array(all_preds)


def plot_final_evaluation(models_dict, loader, device, phase_name):
    print(f"\nStarting Quantitative Test for: {phase_name}")

    num_models = len(models_dict)
    fig, axes = plt.subplots(1, num_models + 1, figsize=(6 * (num_models + 1), 6))
    fig.suptitle(f"Quantitative Results - {phase_name}", fontsize=20, fontweight='bold')

    ax_roc = axes[-1]
    ax_roc.plot([0, 1], [0, 1], 'k--', label='Random')

    colors = ['#aec7e8', '#1f77b4', '#ffbb78', '#ff7f0e', '#98df8a', '#2ca02c']

    for i, (name, model) in enumerate(models_dict.items()):
        y_true, y_probs, y_preds = get_all_predictions(model, loader, device)

        # Metrics
        acc = accuracy_score(y_true, y_preds)
        f1 = f1_score(y_true, y_preds)
        prec = precision_score(y_true, y_preds)
        rec = recall_score(y_true, y_preds)

        print(f"[{name}] Acc: {acc:.4f} | F1: {f1:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f}")

        # Confusion Matrix
        cm = confusion_matrix(y_true, y_preds)
        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            ax=axes[i],
            cbar=False,
            xticklabels=['REAL', 'FAKE'],
            yticklabels=['REAL', 'FAKE']
        )

        axes[i].set_title(f"{name}")
        if i == 0:
            axes[i].set_ylabel('Ground Truth')
        axes[i].set_xlabel('Predicted')

        # ROC Curve
        fpr, tpr, _ = roc_curve(y_true, y_probs)
        roc_auc = auc(fpr, tpr)

        ax_roc.plot(
            fpr,
            tpr,
            color=colors[i],
            lw=2.5,
            label=f"{name} (AUC: {roc_auc:.4f})"
        )

    ax_roc.set_title("ROC Curves")
    ax_roc.set_xlabel("False Positive Rate")
    ax_roc.set_ylabel("True Positive Rate")
    ax_roc.legend(loc="lower right", fontsize=9)

    plt.tight_layout()
    plt.show()

## 6 - Experimental Results

### 6.1 - Intra-Dataset Evaluation

In [ ]:
plot_final_evaluation(trained_models, test_loader, device, "CNN vs Hybrid vs Foundation-CNN (DINOv3)")

### 6.2 - Performance Evaluation: Inference Time and FPS Measurement (CPU vs GPU)

In [ ]:
def measure_inference_time(model, device, img_size=(1, 3, 224, 224), iterations=100):
    model.to(device)
    model.eval()

    # Create a dummy image with the same dimensions as the inputs
    dummy_input = torch.randn(img_size).to(device)

    print(f"--- Starting test on {device.upper()} ---")

    # PHASE 1: Warm-up (necessary to "wake up" the hardware and avoid anomalous spikes)
    with torch.no_grad():
        for _ in range(10):
            _ = model(dummy_input)

    # PHASE 2: Actual measurement
    times = []
    with torch.no_grad():
        for _ in range(iterations):
            # If we are on GPU, we must synchronize before taking the start time
            if device == 'cuda':
                torch.cuda.synchronize()

            start_time = time.perf_counter()

            # Inference
            _ = model(dummy_input)

            # Synchronize at the end as well
            if device == 'cuda':
                torch.cuda.synchronize()

            end_time = time.perf_counter()

            times.append((end_time - start_time) * 1000)

    mean_time = np.mean(times)
    std_time = np.std(times)
    fps = 1000 / mean_time

    print(f"Average time per image: {mean_time:.2f} ms (+/- {std_time:.2f} ms)")
    print(f"Frames Per Second (FPS): {fps:.2f} FPS\n")

    return mean_time, fps

# Benchmarked on the DINOv3 ConvNeXt-Small model (Step 2): the new, heaviest
# architecture introduced in this phase, most relevant to profile.
cpu_time, cpu_fps = measure_inference_time(model_dino_adv, 'cpu')

if torch.cuda.is_available():
    gpu_time, gpu_fps = measure_inference_time(model_dino_adv, 'cuda')
else:
    print("GPU not available on this Colab runtime.")